# Phylogenomic Placement of *Saccharomyces cerevisiae* DoF1

**A step-by-step pipeline for placing a novel wild yeast isolate within the broader *S. cerevisiae* species tree.**

---

## Overview

This notebook walks through a complete phylogenomics pipeline, from raw genome assemblies to a publication-quality phylogenetic tree. We use **DoF1**, a wild *Saccharomyces cerevisiae* isolate collected from the Indiana University Bloomington campus, as our focal strain.

### What is phylogenomics?

Traditional phylogenetics infers evolutionary relationships from one or a few genes. **Phylogenomics** uses hundreds or thousands of genes simultaneously, producing trees with far higher statistical confidence. Instead of picking a single marker (like 18S rRNA), we identify genes shared across all genomes of interest, align them individually, and concatenate the alignments into a single large matrix called a **supermatrix**. A maximum-likelihood algorithm then finds the tree topology that best explains the observed sequence variation.

### Pipeline at a glance

| Step | Tool | What it does |
|------|------|--------------|
| 1 | BUSCO | Finds conserved single-copy genes in each genome |
| 2 | MAFFT | Aligns each shared gene across all strains |
| 3 | trimAl | Removes poorly aligned or gappy columns |
| 4 | Python | Concatenates alignments into a supermatrix |
| 5 | IQ-TREE2 | Infers the maximum-likelihood phylogeny |
| 6 | matplotlib | Visualizes the tree colored by ecological lineage |

### Learning objectives

After working through this notebook, you should be able to:
- Explain what BUSCO scores measure and why they matter for comparative genomics
- Describe the difference between a phylogram and a cladogram
- Interpret bootstrap support values on a phylogenetic tree
- Modify this pipeline to place your own genome assembly in a species tree

---

> **Runtime note:** This notebook is designed for Google Colab. Run cells in order with **Shift+Enter**. The BUSCO step (Cell 5) takes 1–2 hours; all other steps are fast. We recommend mounting Google Drive (Cell 2) so your results survive a session disconnect.

## Cell 1 — Install conda and bioinformatics tools

Google Colab provides a Linux environment with Python pre-installed, but most bioinformatics tools (BUSCO, MAFFT, IQ-TREE) are not available by default. We use **condacolab** to install the conda package manager, which gives us access to the **Bioconda** and **conda-forge** software repositories.

> **Important:** After this cell runs, the Colab runtime will restart automatically. This is normal — it is required for conda to initialize properly. After the restart, **begin at Cell 2** (do not re-run Cell 1).

### Tools we are installing

| Tool | Purpose |
|------|---------|
| `busco=5.8.0` | Benchmarking Universal Single-Copy Orthologs — finds conserved genes |
| `mafft` | Multiple sequence aligner |
| `trimal` | Alignment trimmer |
| `iqtree` | Maximum-likelihood phylogenetic inference |
| `ncbi-datasets-cli` | Command-line tool for downloading NCBI genomes |

In [ ]:
# ── CELL 1 ── Install condacolab and all bioinformatics tools
# The runtime will restart automatically after this cell — that is expected.
# After restart, start at Cell 2.

!pip install -q condacolab
import condacolab
condacolab.install()  # triggers runtime restart

## Cell 2 — Mount Google Drive and install tools

After the runtime restarts, we mount **Google Drive** so that intermediate results (BUSCO outputs, the supermatrix, the tree) are saved persistently. If the Colab session disconnects, we can restore from Drive rather than rerunning hours of computation.

We also install all bioinformatics tools via conda and pin `numpy<2.0`. BUSCO 5.x has a bug where it tries to serialize NumPy `float32` values as JSON, which fails in NumPy 2.x. Pinning to 1.x prevents this error.

In [ ]:
# ── CELL 2 ── Mount Google Drive and install all tools

from google.colab import drive
drive.mount('/content/drive')

import subprocess

print('Installing bioinformatics tools via conda (~10 min)...')
subprocess.run(
    'conda install -y -q -c conda-forge -c bioconda '
    'busco=5.8.0 mafft trimal iqtree ncbi-datasets-cli 2>&1 | tail -5',
    shell=True)

# Pin numpy<2.0 — fixes BUSCO float32 JSON serialization bug
print('Pinning numpy<2.0...')
subprocess.run('pip install -q "numpy<2.0"', shell=True)

# Install Python visualization and bioinformatics libraries
print('Installing Python packages...')
subprocess.run(
    'pip install -q toytree toyplot biopython pandas matplotlib',
    shell=True)

# Verify installations
print('\n=== Tool check ===')
r = subprocess.run('python -c "import numpy; print(numpy.__version__)"',
                   shell=True, capture_output=True, text=True)
print(f'  numpy:   {r.stdout.strip()}  (must be 1.x)')

for tool in ['busco', 'mafft', 'trimal', 'datasets']:
    r = subprocess.run(f'which {tool}', shell=True, capture_output=True)
    print(f'  {tool:12s} {"OK" if r.returncode == 0 else "NOT FOUND"}')

for name in ['iqtree2', 'iqtree']:
    r = subprocess.run(f'which {name}', shell=True, capture_output=True)
    if r.returncode == 0:
        print(f'  {name:12s} OK'); break

print('\nAll tools ready.')

## Cell 3 — Upload the DoF1 genome assembly

This cell uploads the focal genome: the DoF1 *S. cerevisiae* assembly. DoF1 was sequenced with Oxford Nanopore Technology (ONT) long-read sequencing and assembled into ~20 contigs covering ~11.92 Mb — comparable to the ~12 Mb *S. cerevisiae* reference genome.

- **NCBI BioProject:** PRJNA1477971  
- **NCBI BioSample:** SAMN60825368

If you have previously run this notebook and saved the genome to Google Drive, it will be restored automatically. Otherwise, you will be prompted to upload the FASTA file.

In [ ]:
# ── CELL 3 ── Upload DoF1 genome FASTA
# Restores from Google Drive automatically if available.

from google.colab import files
import os, shutil

GENOMES_DIR  = '/content/genomes'
QUERY_DEST   = f'{GENOMES_DIR}/IUB_HOOSIER.fasta'
DRIVE_BACKUP = '/content/drive/MyDrive/DoF1_phylo'
DRIVE_QUERY  = f'{DRIVE_BACKUP}/IUB_HOOSIER.fasta'

os.makedirs(GENOMES_DIR, exist_ok=True)
os.makedirs(DRIVE_BACKUP, exist_ok=True)

if os.path.exists(DRIVE_QUERY):
    shutil.copy(DRIVE_QUERY, QUERY_DEST)
    size = os.path.getsize(QUERY_DEST)
    print(f'Restored DoF1 genome from Drive ({size:,} bytes)')
else:
    print('Upload the DoF1 FASTA file when the picker appears:')
    uploaded = files.upload()
    for fname in uploaded:
        with open(QUERY_DEST, 'wb') as f:
            f.write(uploaded[fname])
        shutil.copy(QUERY_DEST, DRIVE_QUERY)  # back up to Drive
        print(f'Uploaded and backed up to Drive ({len(uploaded[fname]):,} bytes)')

## Cell 4 — Download reference genomes from NCBI

We compare DoF1 against 10 reference *S. cerevisiae* strains representing the major ecological lineages, plus 2 *S. paradoxus* strains as an **outgroup**.

### Why these strains?

*S. cerevisiae* has diversified into distinct populations associated with different human-driven fermentation environments. By sampling strains from each lineage, we can determine which population DoF1 is most closely related to.

| Strain | Lineage | Why it was chosen |
|--------|---------|-------------------|
| S288c | Laboratory | The standard reference genome (Saccharomyces Genome Database) |
| W303, SK1 | Laboratory | Widely used lab strains with distinct genetic backgrounds |
| YJM993 | Clinical | Human clinical isolate; represents pathogenic potential |
| EC1118, AWRI1631 | Wine | Wine fermentation strains; highly studied |
| JAY291 | Bioethanol | Industrial strain from Brazilian sugarcane ethanol |
| RM11-1a | Wild environmental | Wild isolate from a Napa Valley vineyard |
| CLIB89, CLIB324 | Beer | Traditional ale/beer fermentation strains |
| *S. paradoxus* CBS432, N44 | Outgroup | Sister species; used to root the tree |

### What is an outgroup?

An **outgroup** is a taxon that is related to but outside the group of interest (the **ingroup**). By including *S. paradoxus* — the closest relative of *S. cerevisiae* — we can root the tree, meaning we can determine which branch is ancestral. Without an outgroup, we would only know which taxa are related, not the direction of evolution.

### NCBI Datasets

We use the `datasets` command-line tool from NCBI to download genome assemblies by their accession numbers. The `--include genome` flag downloads only the genome FASTA, not annotations.

In [ ]:
# ── CELL 4 ── Download reference genomes from NCBI

import os, subprocess, glob

GENOMES_DIR = '/content/genomes'
os.makedirs(GENOMES_DIR, exist_ok=True)

# Each tuple: (NCBI accession, strain name used in analysis, ecological lineage)
STRAINS = [
    ('GCF_000146045.2', 'S288c',       'laboratory'),
    ('GCA_002163515.1', 'W303',        'laboratory'),
    ('GCA_001524775.2', 'SK1',         'laboratory'),
    ('GCA_000977955.1', 'YJM993',      'clinical'),
    ('GCF_000218975.1', 'EC1118',      'wine'),
    ('GCA_000182765.2', 'AWRI1631',    'wine'),
    ('GCA_000182745.2', 'JAY291',      'bioethanol'),
    ('GCA_000149365.1', 'RM11-1a',     'wild_environmental'),
    ('GCA_003054445.1', 'CLIB89',      'beer'),
    ('GCA_003054405.1', 'CLIB324',     'beer'),
    ('GCA_002079055.1', 'Spar_CBS432', 'outgroup'),   # S. paradoxus
    ('GCF_000292725.1', 'Spar_N44',   'outgroup'),    # S. paradoxus
]

failed = []
for acc, strain, lineage in STRAINS:
    out = f'{GENOMES_DIR}/{strain}.fasta'
    if os.path.exists(out) and os.path.getsize(out) > 10_000:
        print(f'  [skip] {strain} — already downloaded')
        continue

    print(f'  Downloading {strain} ({acc})...', end=' ', flush=True)
    r = subprocess.run(
        f'datasets download genome accession {acc} '
        f'--include genome --filename /tmp/dl.zip',
        shell=True, capture_output=True)

    if r.returncode != 0:
        print('FAILED'); failed.append(strain); continue

    subprocess.run('unzip -o -q /tmp/dl.zip -d /tmp/dl', shell=True)

    # Find the downloaded FASTA and rename all sequence headers to the strain name.
    # This is important: BUSCO will use the filename as the strain identifier.
    fna = subprocess.run(
        'find /tmp/dl -name "*.fna" | head -1',
        shell=True, capture_output=True, text=True).stdout.strip()

    if fna:
        # Rename FASTA headers: >strain_originalHeader
        subprocess.run(
            f"awk -v s={strain} '/^>/{{print \">\"s\"_\"substr($0,2);next}}{{print}}' "
            f"{fna} > {out}", shell=True)
        print('OK')
    else:
        print('no FASTA found'); failed.append(strain)

    subprocess.run('rm -rf /tmp/dl /tmp/dl.zip', shell=True)

fastas = glob.glob(f'{GENOMES_DIR}/*.fasta')
print(f'\n{len(fastas)} genome FASTA files ready in {GENOMES_DIR}/')
if failed:
    print(f'Failed downloads: {failed}')

## Cell 5 — Run BUSCO

**BUSCO** (Benchmarking Universal Single-Copy Orthologs) identifies conserved genes that are expected to be present in a single copy in all genomes of a given lineage. We use the `saccharomycetes_odb10` lineage dataset, which contains **2,137 marker genes** shared across the Saccharomycetes class of fungi.

### Why BUSCO for phylogenomics?

For a phylogenomic supermatrix, we need genes that are:
1. **Present in all genomes** — so every taxon has a sequence to contribute
2. **Single-copy** — duplicated genes would introduce paralogs that could mislead the tree
3. **Conserved enough to align** — but variable enough to carry phylogenetic signal

BUSCO's single-copy orthologs satisfy all three criteria, making them ideal phylogenetic markers.

### BUSCO output categories

| Category | Meaning |
|----------|---------|
| **C (Complete)** | Gene found in expected copy number |
| **S (Single-copy)** | Complete and present once — ideal for phylogenomics |
| **D (Duplicated)** | Complete but found in multiple copies |
| **F (Fragmented)** | Partial match found |
| **M (Missing)** | Not found |

We want high **S** scores (>85%) for all genomes. Genomes with low completeness are dropped before supermatrix construction.

### Parameters used

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `-l saccharomycetes_odb10` | lineage dataset | Use the 2,137-gene yeast marker set |
| `-m genome` | mode | Input is a genome assembly (not proteins or transcriptome) |
| `--cpu 2` | threads | Use 2 CPU cores (Colab free tier limit) |
| `--force` | overwrite | Redo even if output exists (ensures clean run) |

In [ ]:
# ── CELL 5 ── Run BUSCO on all genomes
# This step takes 1–2 hours. Results are printed as each genome finishes.

import os, glob, subprocess

GENOMES_DIR = '/content/genomes'
BUSCO_DIR   = '/content/busco_results'
LINEAGE     = 'saccharomycetes_odb10'  # 2,137 conserved yeast genes
os.makedirs(BUSCO_DIR, exist_ok=True)

for fasta in sorted(glob.glob(f'{GENOMES_DIR}/*.fasta')):
    strain = os.path.basename(fasta).replace('.fasta', '')

    # Skip if already completed (allows resuming after disconnects)
    done = glob.glob(f'{BUSCO_DIR}/{strain}/short_summary*.txt')
    if done:
        print(f'  [skip] {strain} — BUSCO already complete')
        continue

    print(f'  Running BUSCO: {strain}...')
    subprocess.run(
        f'busco '
        f'-i {fasta} '              # input genome
        f'-o {strain} '             # output directory name
        f'--out_path {BUSCO_DIR} '  # parent directory for output
        f'-l {LINEAGE} '            # lineage marker set
        f'-m genome '               # genome mode
        f'--cpu 2 '                 # threads
        f'--force '                 # overwrite existing output
        f'2>&1 | tail -5',          # show only last 5 lines of progress
        shell=True)

    # Print the completeness summary for this genome
    summary = glob.glob(f'{BUSCO_DIR}/{strain}/short_summary*.txt')
    if summary:
        for line in open(summary[0]):
            if 'C:' in line:
                print(f'    Result: {line.strip()}')
                break
    else:
        print(f'    WARNING: No summary found — BUSCO may have failed for {strain}')

# ── Summary table ──
print('\n' + '='*60)
print('BUSCO completeness summary')
print('='*60)
low_strains = []
for s in sorted(glob.glob(f'{BUSCO_DIR}/*/short_summary*.txt')):
    strain = s.split('/')[-2]
    for line in open(s):
        if 'C:' in line:
            pct = float(line.split('C:')[1].split('%')[0])
            flag = '  *** LOW — consider dropping' if pct < 80 else ''
            print(f'  {strain:20s}  {line.strip()}{flag}')
            if pct < 80:
                low_strains.append(strain)
            break

if low_strains:
    print(f'\nStrains with <80% completeness: {low_strains}')
    print('Remove these from /content/busco_results/ and /content/genomes/ before Cell 7.')

## Cell 6 — Back up BUSCO results to Google Drive

BUSCO results are large (~500 MB) and took hours to compute. This cell saves them to your Google Drive so you can restore them after a runtime disconnect without re-running BUSCO.

It also downloads a small summary table to your desktop.

In [ ]:
# ── CELL 6 ── Back up BUSCO results to Google Drive

import subprocess, shutil, os, glob
from google.colab import files

DRIVE_BACKUP = '/content/drive/MyDrive/DoF1_phylo'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

print('Archiving BUSCO results to Drive (~2 min)...')
subprocess.run(
    f'tar -czf {DRIVE_BACKUP}/busco_results.tar.gz -C /content busco_results',
    shell=True)
size = os.path.getsize(f'{DRIVE_BACKUP}/busco_results.tar.gz')
print(f'Saved busco_results.tar.gz ({size/1e6:.1f} MB) to Drive')

# Write a plain-text completeness table and download it
lines = ['strain\tcompleteness\n']
for s in sorted(glob.glob('/content/busco_results/*/short_summary*.txt')):
    strain = s.split('/')[-2]
    for line in open(s):
        if 'C:' in line:
            lines.append(f'{strain}\t{line.strip()}\n')
            break
with open('/content/busco_summary.tsv', 'w') as f:
    f.writelines(lines)
files.download('/content/busco_summary.tsv')
print('Downloaded busco_summary.tsv.')

## Cell 6-RESTORE — Restore BUSCO results after a runtime restart

If your Colab session disconnected after completing BUSCO, run this cell instead of re-running Cell 5. It restores the archived results from Google Drive.

In [ ]:
# ── CELL 6-RESTORE ── Restore BUSCO results from Drive (run after runtime restart)

import subprocess, os

DRIVE_BACKUP = '/content/drive/MyDrive/DoF1_phylo'
archive      = f'{DRIVE_BACKUP}/busco_results.tar.gz'

if os.path.exists(archive):
    print('Restoring BUSCO results from Drive...')
    subprocess.run(f'tar -xzf {archive} -C /content', shell=True)
    import glob
    n = len(glob.glob('/content/busco_results/*'))
    print(f'Restored {n} BUSCO result directories.')
else:
    print(f'No backup found at {archive} — run Cells 3–5 first.')

## Cell 7 — Extract shared single-copy orthologs

BUSCO writes a FASTA file for each single-copy gene it finds in each genome. Now we need to identify the genes that are present as a single copy in **all** genomes simultaneously — these are the markers we can use in the supermatrix.

### The intersection problem

If Genome A has genes {1, 2, 3, 4, 5} and Genome B has genes {2, 3, 4, 5, 6}, the shared set is {2, 3, 4, 5}. We take the intersection across all 13 genomes. Because no genome achieves 100% BUSCO completeness, the shared set is always smaller than the full marker set.

### Random subsampling to 500 orthologs

With ~1,800+ shared orthologs, the concatenated supermatrix would be ~1 million amino acid sites, making IQ-TREE very slow. We randomly subsample **500 orthologs** (with a fixed random seed for reproducibility). Studies show that 500+ orthologs provide sufficient resolution for within-species phylogenomics — adding more genes does not change the topology, only marginally improves bootstrap values.

In [ ]:
# ── CELL 7 ── Extract shared single-copy orthologs

import os, glob, random
from pathlib import Path

BUSCO_DIR = '/content/busco_results'
ORTHO_DIR = '/content/orthologs'
os.makedirs(ORTHO_DIR, exist_ok=True)

N_ORTHOLOGS = 500    # number of orthologs to subsample for the supermatrix
RANDOM_SEED = 42     # fixed seed ensures reproducibility

strains = sorted([d for d in os.listdir(BUSCO_DIR)
                  if os.path.isdir(f'{BUSCO_DIR}/{d}')])
print(f'Strains found: {strains}\n')

def get_busco_seq_dir(strain):
    """Find the single_copy_busco_sequences directory.
    The run_* subdirectory name varies by BUSCO version, so we use glob."""
    hits = glob.glob(
        f'{BUSCO_DIR}/{strain}/run_*/busco_sequences/single_copy_busco_sequences')
    return hits[0] if hits else None

# Get the set of single-copy BUSCO IDs for each strain
strain_ids = {}
for s in strains:
    seq_dir = get_busco_seq_dir(s)
    ids = {Path(f).stem for f in glob.glob(f'{seq_dir}/*.faa')} if seq_dir else set()
    strain_ids[s] = ids
    print(f'  {s:20s}  {len(ids)} single-copy BUSCOs')

# Intersection: genes present as single-copy in ALL strains
all_shared = set.intersection(*strain_ids.values())
print(f'\nTotal shared single-copy orthologs: {len(all_shared)}')

# Randomly subsample for speed
random.seed(RANDOM_SEED)
selected = random.sample(sorted(all_shared), min(N_ORTHOLOGS, len(all_shared)))
print(f'Subsampled to {len(selected)} orthologs (seed={RANDOM_SEED})')

# Write one multi-FASTA per ortholog, with one sequence per strain
for oid in sorted(selected):
    with open(f'{ORTHO_DIR}/{oid}.faa', 'w') as out:
        for strain in strains:
            seq_dir = get_busco_seq_dir(strain)
            seqs = [l for l in open(f'{seq_dir}/{oid}.faa') if not l.startswith('>')]
            out.write(f'>{strain}\n')
            out.writelines(seqs)

print(f'\nWrote {len(selected)} ortholog FASTAs to {ORTHO_DIR}/')

## Cell 8 — Align, trim, and concatenate into a supermatrix

### Step 1: Multiple sequence alignment with MAFFT

Each ortholog FASTA contains one protein sequence per strain. Amino acid positions that are homologous (descended from the same ancestral position) must be lined up before we can compare them. **MAFFT** with `--auto` automatically selects the best alignment algorithm based on sequence length and number of sequences.

### Step 2: Trim with trimAl

Aligned sequences often contain **gap-rich columns** where alignment is uncertain. These columns add noise rather than signal. **trimAl** with `-gappyout` automatically removes the most gap-heavy columns using a statistical approach. This is a conservative filter that preserves most of the alignment.

### Step 3: Concatenate into a supermatrix

After trimming, we join all 500 alignments end-to-end into a single matrix called a **supermatrix** (also called a concatenated alignment). Each row is a strain; each column is a homologous amino acid position from one of the 500 genes. The supermatrix is the input to IQ-TREE.

```
          [Gene 1 alignment] [Gene 2 alignment] ... [Gene 500 alignment]
DoF1:     MSTLKV...          AGVTIK...              PLKRST...
S288c:    MSTLKV...          AGVTIK...              PLKRST...
YJM993:   MSTLKV...          AGVTIQ...              PLKRAT...
...
```

In [ ]:
# ── CELL 8 ── Align (MAFFT) + trim (trimAl) + concatenate into supermatrix

import os, glob, subprocess, shutil
from collections import OrderedDict
from pathlib import Path

ORTHO_DIR   = '/content/orthologs'
ALIGNED_DIR = '/content/aligned'
TRIMMED_DIR = '/content/trimmed'
CONCAT_DIR  = '/content/concat'
for d in [ALIGNED_DIR, TRIMMED_DIR, CONCAT_DIR]:
    os.makedirs(d, exist_ok=True)

orthos = sorted(glob.glob(f'{ORTHO_DIR}/*.faa'))
print(f'Aligning and trimming {len(orthos)} orthologs...')

for i, fasta in enumerate(orthos):
    oid  = Path(fasta).stem
    aln  = f'{ALIGNED_DIR}/{oid}.aln'
    trim = f'{TRIMMED_DIR}/{oid}.trim'

    # Skip if already processed
    if os.path.exists(trim) and os.path.getsize(trim) > 0:
        continue

    # Step 1: align with MAFFT
    # --auto: selects algorithm automatically (FFT-NS-2 for speed, L-INS-i for accuracy)
    # --quiet: suppress progress output
    subprocess.run(f'mafft --auto --quiet {fasta} > {aln}', shell=True)

    # Step 2: trim with trimAl
    # -gappyout: removes columns with exceptionally high gap rates
    # -fasta: output in FASTA format
    subprocess.run(
        f'trimal -in {aln} -out {trim} -gappyout -fasta',
        shell=True, capture_output=True)

    # Fall back to untrimmed alignment if trimAl removes all columns
    if not os.path.exists(trim) or os.path.getsize(trim) == 0:
        shutil.copy(aln, trim)

    if (i + 1) % 100 == 0:
        print(f'  Processed {i+1}/{len(orthos)} orthologs...')

print('Alignments complete. Building supermatrix...')

# ── Step 3: Concatenate into supermatrix ──
trim_files  = sorted(glob.glob(f'{TRIMMED_DIR}/*.trim'))
all_taxa    = None
supermatrix = OrderedDict()  # taxon -> list of alignment blocks
partitions  = []             # (gene_name, start_pos, end_pos)
pos = 1

for tf in trim_files:
    # Parse the trimmed FASTA
    seqs = {}
    header = None
    with open(tf) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('>'): header = line[1:]; seqs[header] = []
            elif header: seqs[header].append(line)
    seqs = {k: ''.join(v) for k, v in seqs.items()}
    if not seqs: continue

    # Initialize taxon list from first file
    if all_taxa is None:
        all_taxa = set(seqs.keys())
        for t in all_taxa: supermatrix[t] = []

    # Skip gene if it is missing any taxon
    if set(seqs.keys()) != all_taxa: continue

    length = len(next(iter(seqs.values())))
    partitions.append((Path(tf).stem, pos, pos + length - 1))
    pos += length
    for t in all_taxa:
        supermatrix[t].append(seqs[t])

# Write the supermatrix FASTA
with open(f'{CONCAT_DIR}/supermatrix.faa', 'w') as f:
    for taxon, parts in supermatrix.items():
        f.write(f'>{taxon}\n{"" .join(parts)}\n')

# Write partition file (records gene boundaries — not used here but useful for record-keeping)
with open(f'{CONCAT_DIR}/partitions.txt', 'w') as f:
    for name, start, end in partitions:
        f.write(f'AA, {name} = {start}-{end}\n')

print(f'Supermatrix: {len(supermatrix)} taxa × {pos-1:,} amino acid sites')
print(f'Written to: {CONCAT_DIR}/supermatrix.faa')

## Cell 9 — Back up supermatrix to Google Drive

The supermatrix (~3 MB) takes ~30 minutes to generate. This cell saves it to Drive so we can restore it without redoing the alignment step.

In [ ]:
# ── CELL 9 ── Back up supermatrix to Google Drive

import shutil, os
from google.colab import files

DRIVE_BACKUP = '/content/drive/MyDrive/DoF1_phylo'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

for fname in ['supermatrix.faa', 'partitions.txt']:
    src = f'/content/concat/{fname}'
    shutil.copy(src, f'{DRIVE_BACKUP}/{fname}')
    print(f'Saved {fname} to Drive')

files.download('/content/concat/supermatrix.faa')
print('Downloaded supermatrix.faa to desktop.')

## Cell 9-RESTORE — Restore supermatrix after a runtime restart

In [ ]:
# ── CELL 9-RESTORE ── Restore supermatrix from Drive (run after runtime restart)

import shutil, os

DRIVE_BACKUP = '/content/drive/MyDrive/DoF1_phylo'
os.makedirs('/content/concat', exist_ok=True)

for fname in ['supermatrix.faa', 'partitions.txt']:
    src = f'{DRIVE_BACKUP}/{fname}'
    dst = f'/content/concat/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Restored {fname} ({os.path.getsize(dst)/1e6:.1f} MB)')
    else:
        print(f'Not found: {fname} — run Cells 7-8 first.')

## Cell 10 — Maximum-likelihood phylogenetic inference with IQ-TREE2

**IQ-TREE2** is a state-of-the-art program for maximum-likelihood phylogenetic inference. It finds the tree topology and branch lengths that maximize the probability of observing the sequence data, given a specified evolutionary model.

### The evolutionary model: LG+G4

We use the **LG+G4** amino acid substitution model:

- **LG** (Le & Gascuel, 2008): An empirical matrix describing the rates at which each amino acid substitutes for each other (e.g., how often Alanine → Valine). This was derived from a large database of alignments and is the standard model for multi-gene protein phylogenomics.
- **+G4**: Gamma rate variation with 4 rate categories. Different positions in a protein evolve at different rates — some are under strong functional constraint (slowly evolving), others are hypervariable (fast evolving). The Gamma distribution models this variation.

### Bootstrap support

We run **500 ultrafast bootstrap** replicates (`-B 500 --bnni`). Bootstrap values indicate how reproducibly a clade is recovered when we resample the data:

| Bootstrap value | Interpretation |
|----------------|----------------|
| ≥95% | Strong support — clade is very reliable |
| 70–94% | Moderate support — likely correct |
| <70% | Weak support — interpret with caution |

### Key parameters

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `-s` | supermatrix.faa | Input alignment |
| `-m LG+G4` | model | Evolutionary model (skips ModelFinder for speed) |
| `-B 500` | bootstrap reps | 500 ultrafast bootstrap replicates |
| `--bnni` | optimization | Further optimize bootstrap trees (improves accuracy) |
| `-T 1` | threads | Single thread (avoids parallelization hangs on Colab) |
| `--redo` | overwrite | Force rerun if output files exist |

In [ ]:
# ── CELL 10 ── IQ-TREE2: maximum-likelihood tree inference
# Runs in the background and polls the log file every 60 seconds.
# Expected runtime: 30–90 minutes.

import subprocess, os, time

CONCAT_DIR = '/content/concat'
IQTREE_DIR = '/content/iqtree'
os.makedirs(IQTREE_DIR, exist_ok=True)

# Find the IQ-TREE binary (may be named iqtree or iqtree2)
iqtree_bin = 'iqtree'
for name in ['iqtree2', 'iqtree']:
    r = subprocess.run(f'which {name}', shell=True, capture_output=True)
    if r.returncode == 0:
        iqtree_bin = name
        break
print(f'IQ-TREE binary: {iqtree_bin}')

cmd = (
    f'{iqtree_bin} '
    f'-s {CONCAT_DIR}/supermatrix.faa '   # supermatrix input
    f'-m LG+G4 '                           # evolutionary model
    f'-B 500 --bnni '                      # 500 ultrafast bootstrap + optimization
    f'-T 1 '                               # single thread (stable on Colab)
    f'--prefix {IQTREE_DIR}/DoF1_phylo '  # output file prefix
    f'--redo '                             # overwrite existing output
    f'> {IQTREE_DIR}/iqtree.log 2>&1'     # redirect all output to log file
)

print('Starting IQ-TREE... (log updates every 60 s)')
proc = subprocess.Popen(cmd, shell=True)

# Poll the log file so we can see progress without capturing output
log_file = f'{IQTREE_DIR}/iqtree.log'
last_lines_shown = 0
while proc.poll() is None:
    time.sleep(60)
    if os.path.exists(log_file):
        with open(log_file) as lf:
            lines = lf.readlines()
        new_lines = lines[last_lines_shown:]
        for line in new_lines:
            line = line.strip()
            # Print informative lines only
            if any(kw in line for kw in
                   ['Log-likelihood', 'Bootstrap', 'BEST', 'Iteration',
                    'seconds', 'minutes', 'hour', 'ERROR']):
                print(f'  {line}')
        last_lines_shown = len(lines)

# Check output
treefile = f'{IQTREE_DIR}/DoF1_phylo.treefile'
if os.path.exists(treefile):
    print('\nIQ-TREE complete! Newick tree:')
    with open(treefile) as f:
        print(f.read())
else:
    print('\nERROR: No treefile produced. Check iqtree.log for details.')
    with open(log_file) as f:
        print(f.read()[-2000:])

## Cell 11 — Back up IQ-TREE results to Google Drive

Saves the treefile and log to Drive and downloads the treefile to your desktop.

In [ ]:
# ── CELL 11 ── Back up IQ-TREE results to Google Drive

import shutil, os, glob
from google.colab import files

DRIVE_BACKUP = '/content/drive/MyDrive/DoF1_phylo'
IQTREE_DIR   = '/content/iqtree'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

for f in glob.glob(f'{IQTREE_DIR}/DoF1_phylo.*'):
    shutil.copy(f, DRIVE_BACKUP)
    print(f'Saved to Drive: {os.path.basename(f)}')

files.download(f'{IQTREE_DIR}/DoF1_phylo.treefile')
print('Downloaded DoF1_phylo.treefile to desktop.')

## Cell 11-RESTORE — Restore IQ-TREE results after a runtime restart

If you are returning to the notebook after a disconnect and just want to re-run the visualization, use this cell to restore the treefile from Drive. Then go directly to Cell 12.

In [ ]:
# ── CELL 11-RESTORE ── Restore IQ-TREE results from Drive

import shutil, os, glob

DRIVE_BACKUP = '/content/drive/MyDrive/DoF1_phylo'
IQTREE_DIR   = '/content/iqtree'
os.makedirs(IQTREE_DIR, exist_ok=True)

backed = glob.glob(f'{DRIVE_BACKUP}/DoF1_phylo.*')
for f in backed:
    shutil.copy(f, IQTREE_DIR)
    print(f'Restored: {os.path.basename(f)}')
if not backed:
    print('No IQ-TREE files found in Drive — run Cells 9-10 first.')

## Cell 12 — Visualize the phylogenetic tree

### Rooting

IQ-TREE produces an **unrooted** tree — it finds the topology and branch lengths but does not specify which end of the tree is ancestral. We root the tree on the **midpoint** between the *S. paradoxus* outgroup strains (CBS432 and N44), which are the most distantly related taxa in our dataset. This makes biological sense: *S. paradoxus* diverged from *S. cerevisiae* ~1–5 million years ago.

### Cladogram vs. phylogram

This tree is drawn as a **cladogram** (all branch lengths equal) rather than a **phylogram** (branch lengths proportional to evolutionary change). One of our strains (AWRI1631) has an unusually long branch that would compress all other branches into invisibility in a phylogram. A cladogram emphasizes topology — which taxa are related — at the expense of branch length information. The true branch lengths are preserved in the `.nwk` file.

### Reading the tree

- **Nodes** (branch points) represent common ancestors
- **Bootstrap values** at nodes indicate support: 100 = perfectly reproducible, <70 = unreliable
- **Tip labels** are the strains; color indicates ecological lineage
- **DoF1** (gold star ★) is the focal strain of this study

In [ ]:
# ── CELL 12 ── Publication-quality phylogenetic tree visualization

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import toytree, os
from Bio import Phylo
from collections import Counter
from pathlib import Path

IQTREE_DIR = '/content/iqtree'
TREEFILE   = f'{IQTREE_DIR}/DoF1_phylo.treefile'
OUT_NWK    = f'{IQTREE_DIR}/DoF1_phylo.rooted.nwk'
OUT_PNG    = f'{IQTREE_DIR}/DoF1_phylo_tree.png'
OUT_TXT    = f'{IQTREE_DIR}/DoF1_placement_summary.txt'
OUTGROUP   = ['Spar_CBS432', 'Spar_N44']

# Ecological lineage assignment for each strain
STRAIN_LINEAGE = {
    'IUB_HOOSIER':  'wild_environmental',
    'S288c':        'laboratory',
    'W303':         'laboratory',
    'SK1':          'laboratory',
    'YJM993':       'clinical',
    'EC1118':       'wine',
    'AWRI1631':     'wine',
    'JAY291':       'bioethanol',
    'RM11-1a':      'wild_environmental',
    'CLIB89':       'beer',
    'CLIB324':      'beer',
    'Spar_CBS432':  'outgroup',
    'Spar_N44':     'outgroup',
}

# Colors for each lineage
LINEAGE_COLORS = {
    'wild_environmental': '#2196F3',  # blue
    'wine':               '#9C27B0',  # purple
    'bioethanol':         '#FF9800',  # orange
    'beer':               '#795548',  # brown
    'laboratory':         '#607D8B',  # grey-blue
    'clinical':           '#F44336',  # red
    'outgroup':           '#757575',  # grey
}
IUB_COLOR = '#FFD600'  # gold — distinctive highlight for focal strain

# Human-readable tip labels
DISPLAY_NAMES = {
    'IUB_HOOSIER':  'DoF1 ★',             # star marks this study's strain
    'S288c':        'S288c',
    'W303':         'W303',
    'SK1':          'SK1',
    'YJM993':       'YJM993',
    'EC1118':       'EC1118',
    'AWRI1631':     'AWRI1631',
    'JAY291':       'JAY291',
    'RM11-1a':      'RM11-1a',
    'CLIB89':       'CLIB89',
    'CLIB324':      'CLIB324',
    'Spar_CBS432':  'S. paradoxus CBS432',
    'Spar_N44':     'S. paradoxus N44',
}
LINEAGE_LABELS = {
    'wild_environmental': 'Wild environmental',
    'wine':               'Wine',
    'bioethanol':         'Bioethanol',
    'beer':               'Beer',
    'laboratory':         'Laboratory',
    'clinical':           'Clinical',
    'outgroup':           'Outgroup (S. paradoxus)',
}

# ── 1. Root the tree on S. paradoxus using toytree ──
tr = toytree.tree(TREEFILE)
og = [t for t in OUTGROUP if t in tr.get_tip_labels()]
if og:
    mrca = tr.get_mrca_node(*og)
    tr   = tr.root(mrca)
    print(f'Rooted on: {og}')
tr.write(OUT_NWK)
print(f'Rooted Newick saved: {OUT_NWK}')

# ── 2. Load with Bio.Phylo ──
# Keep two copies: one with real branch lengths (for summary stats),
# one with uniform lengths (for display — avoids long-branch compression)
bio_tree_real    = Phylo.read(OUT_NWK, 'newick')
bio_tree_display = Phylo.read(OUT_NWK, 'newick')
for clade in bio_tree_display.find_clades():
    clade.branch_length = 1.0  # uniform lengths = cladogram display

n = bio_tree_display.count_terminals()
print(f'Leaves in tree: {n}')

# ── 3. Build label color dictionary (keyed by display name) ──
label_colors = {}
for raw, display in DISPLAY_NAMES.items():
    if raw == 'IUB_HOOSIER':
        label_colors[display] = IUB_COLOR
    else:
        lineage = STRAIN_LINEAGE.get(raw, '')
        label_colors[display] = LINEAGE_COLORS.get(lineage, '#333333')

# ── 4. Bootstrap label function: show values >=70 on internal nodes only ──
def bootstrap_label(clade):
    if clade.is_terminal():
        return ''
    try:
        val = int(float(clade.confidence))
        return str(val) if val >= 70 else ''
    except (TypeError, ValueError):
        return ''

# ── 5. Tip label function: map internal IDs to display names ──
def tip_label(clade):
    return DISPLAY_NAMES.get(clade.name, clade.name or '')

# ── 6. Draw the tree ──
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size':   11,
})

fig, ax = plt.subplots(figsize=(14, max(8, n * 0.62)))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

Phylo.draw(
    bio_tree_display,
    axes=ax,
    do_show=False,
    label_colors=label_colors,
    branch_labels=bootstrap_label,
    label_func=tip_label,
)

# ── 7. Post-draw: style specific labels ──
for txt in ax.texts:
    content = txt.get_text()
    if 'DoF1' in content:
        txt.set_fontsize(12.5)
        txt.set_fontweight('bold')
        txt.set_path_effects([
            pe.withStroke(linewidth=2.5, foreground='white'),
            pe.Normal(),
        ])
    elif 'paradoxus' in content:
        txt.set_style('italic')  # species name convention
    elif content.isdigit():
        txt.set_fontsize(8.5)
        txt.set_color('#555555')

# ── 8. Clean up axes ──
for spine in ['top', 'right', 'left', 'bottom']:
    ax.spines[spine].set_visible(False)
ax.set_yticks([])
ax.set_xticks([])
ax.set_xlabel('')
ax.set_ylabel('')

# ── 9. Legend ──
legend_handles = [
    mpatches.Patch(facecolor=LINEAGE_COLORS[lin], edgecolor='#ccc',
                   label=LINEAGE_LABELS[lin])
    for lin in LINEAGE_LABELS
] + [
    mpatches.Patch(facecolor=IUB_COLOR, edgecolor='#aaa',
                   label='DoF1, this study')
]
ax.legend(
    handles=legend_handles,
    loc='lower left',
    fontsize=9.5,
    framealpha=0.92,
    edgecolor='#cccccc',
    title='Lineage',
    title_fontsize=10,
)

# ── 10. Key result annotation box ──
ax.annotate(
    'DoF1 sister to YJM993\n(clinical isolate; 91% bootstrap)',
    xy=(0.99, 0.04), xycoords='axes fraction',
    ha='right', va='bottom', fontsize=9,
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFFDE7',
              edgecolor='#FFD600', linewidth=1.2),
)

# ── 11. Caption ──
fig.text(
    0.5, 0.005,
    'Cladogram (branch lengths equalized for display; see .nwk for actual lengths). '
    'Bootstrap values ≥70 shown at nodes. '
    'Tree inferred with IQ-TREE2 / LG+G4 / 500 shared single-copy BUSCOs.',
    ha='center', va='bottom', fontsize=7.5, color='#666666', style='italic',
)

plt.tight_layout(rect=[0, 0.018, 1, 1])
plt.savefig(OUT_PNG, format='png', bbox_inches='tight', dpi=300)
print(f'PNG -> {OUT_PNG}')
plt.show()

# ── 12. Placement summary ──
try:
    target = next(c for c in bio_tree_real.find_clades() if c.name == 'IUB_HOOSIER')
    parent = next(c for c in bio_tree_real.find_clades() if target in c.clades)
    sisters = [c.name for c in parent.clades if c.name and c.name != 'IUB_HOOSIER']
    sister_conf = None
    try:
        sister_conf = int(float(parent.confidence))
    except Exception:
        pass
    clade_taxa    = [t.name for t in parent.get_terminals() if t.name != 'IUB_HOOSIER']
    lineage_count = Counter(STRAIN_LINEAGE.get(t, 'unknown') for t in clade_taxa)
    dominant      = lineage_count.most_common(1)[0]
    summary = (
        f'DoF1 Phylogenetic Placement Summary\n'
        f'{"="*54}\n'
        f'Sister taxon(a):    {sisters}\n'
        f'Bootstrap support:  {sister_conf}%\n'
        f'Sister lineage(s):  {list(set(STRAIN_LINEAGE.get(s,"?") for s in sisters))}\n'
        f'Dominant lineage:   {dominant[0]} ({dominant[1]}/{len(clade_taxa)} strains)\n'
        f'Branch length:      {target.branch_length:.6f} aa subst./site\n'
        f'\nInterpretation: DoF1 groups with {dominant[0].replace("_"," ")} '
        f'strains with {sister_conf}% bootstrap support.\n'
        f'Its short branch length ({target.branch_length:.6f}) indicates minimal '
        f'divergence from the reference population.\n'
    )
    print('\n' + summary)
    Path(OUT_TXT).write_text(summary)
except Exception as e:
    print(f'Placement summary error: {e}')

## Cell 13 — Download all outputs

Downloads all final output files to your desktop and backs them up to Google Drive.

In [ ]:
# ── CELL 13 ── Download all outputs to desktop

from google.colab import files
import shutil, os

DRIVE_BACKUP = '/content/drive/MyDrive/DoF1_phylo'
IQTREE_DIR   = '/content/iqtree'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

outputs = {
    f'{IQTREE_DIR}/DoF1_phylo.rooted.nwk':    'Rooted Newick tree',
    f'{IQTREE_DIR}/DoF1_phylo_tree.png':       'Publication figure (PNG 300 dpi)',
    f'{IQTREE_DIR}/DoF1_placement_summary.txt':'Placement summary',
    f'{IQTREE_DIR}/DoF1_phylo.log':            'IQ-TREE log (methods detail)',
}

for path, desc in outputs.items():
    if os.path.exists(path):
        shutil.copy(path, f'{DRIVE_BACKUP}/{os.path.basename(path)}')
        files.download(path)
        print(f'Downloaded: {os.path.basename(path)}  ({desc})')
    else:
        print(f'Not found:  {os.path.basename(path)} — run Cell 12 first')

print('\nDone. Check your Downloads folder.')